From the youtube videos class. Since I am trying to solve a game that is played in a space like battleship I think that taking his heatmap idea would be a great idea. My hypothesis for how it would look is the star position would be the most played first move with the 3,4 and 4,3 in each corner being next if not the star point. The order would mostlikely be Corner --> Edge --> Fight where there is competition --> Center. This hypothesis would only work for the 19x19 because I really only know that board compared to the 9x9 and 11x11 boards.

For a given strategy I would try to identify something similar to "influence" in a given space and assign it a vector/direction like a vertical wall will have influence to the horizontal sides of it where there are little stones fromn the other player. I should do some research as to how much influence a single piece has alone and where it has more or less on an empty board.

For a strategy of Go. To make it simple I will create one of a few choices where you analyze the board state based on the last move played by the other player. Baseline is that players will place a stone to increase their influence and liberties while decreasing your influence and liberties. Think in micro vs. macro moves always asses the current fights on the board and see if you are up or down liberties compared to the other opponent in each fight and prioritize fights with the most influence then pieces at risk.

Board reduction: Symmetry of the board in rotation


Lecturing about connect4:
You need a clean way of representing your state so then your reward function can get you more uses in more states

Reward Function(State, Action). Example: 4(W) in a row gets tons of points, 3(W) + Empty gets slightly less, 2(w) + Empty gets slightly less, 4(B) gets large negative points, 3(B) + Empty gets slighlty less negative, etc. To evaluate a move you can make computer go 3 moves in the future

My first strategy for a reward function for GO of specifically 9x9: Agressive

Rewards:

Decreasing amount of points starting with the middle
If captures opponent pieces gets the most points
Rid of liberty of opponent and move is closest to same color massive points
Creates a connected structure of two eyes medium amount of points
Placeing piece in a corner spot will give large negative points
Placing a piece on board of board will give medium negtative points



I'm building a 9x9 Go agent in Python for a computational economics class, and I need a reward function that scores candidate moves, which I'll keep changing as I experiment. [Paste your board code here, or say you need a minimal Board class with liberty and capture logic.] My first strategy, "Aggressive," has six rules. Moves earn a bonus that shrinks with distance from the center point (4,4). Capturing opponent stones earns the largest reward, scaled by the number of stones captured. A move that removes an opponent group's liberty while sitting next to one of my own stones earns a massive reward. Creating a connected group with two eyes earns a medium reward. Corner moves get a large penalty, and other edge moves get a medium penalty. Before coding, ask me whether captures or the liberty-and-connection rule should rank higher, and whether corner moves should get both penalties. Write each rule as its own small function, and keep all weights in a separate config (a dict or YAML file), so I can change numbers, toggle rules on or off, or add new strategies without editing the logic. Make `compute_reward(board, move, color, strategy)` return both the total and a per-rule breakdown so I can debug scores. Handle illegal moves clearly, and leave a hook for a final win/loss reward. Include pytest tests that use small hand-built positions to check each rule. Finally, show me the score breakdown for three example moves so I can sanity-check the weights.

## Engine update

What changed from the first version:
- **Ko rule added.** The old code let both players retake the same stone forever. (Your test game shows it: White 3 4, Black 3 3, White 3 4, ...) Test 4 below replays that exact game.
- **`play_move` returns a `MoveResult`.** It still works as True/False, and it also gives the reason a move was illegal (`"suicide"`, `"ko"`, ...) and the list of **captured stones**, which the reward function needs.
- **Atari and eye helpers:** `is_in_atari`, `groups_in_atari`, `is_eye`, `count_eyes`, `has_two_eyes`.
- **Solver tools:** `copy()`, `is_legal()` (checks a move without playing it), `legal_moves()`.
- **Scoring:** `score()` and `winner()` use area scoring with komi 7.5, which fills in your old TODO.
- **Faster:** neighbors are precomputed, and the group and its liberties come from one search. Illegal moves are undone without copying the whole board. In random games this runs about 7–8x faster than the first version.
- **No `print` inside the engine**, so a solver can test thousands of moves silently.


In [ ]:
# ---------- Load the engine + reward code from the .py files ----------
# The engine now lives in go_engine.py (moved out of this cell unchanged),
# the reward function in reward.py, and the weights in strategies.py.
# Locally: run the notebook from the project folder.
# In Colab: the files are fetched from GitHub (push them first).
import os, subprocess, sys

if "google.colab" in sys.modules and not os.path.exists("go_engine.py"):
    if not os.path.exists("GOsolver"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/dereka-code/GOsolver.git"], check=True)
    sys.path.insert(0, "GOsolver")

from go_engine import *

In [2]:
# ---------- Rule tests: run this cell; it should print "All tests passed" ----------

def setup(rows):
    """Build a game from a picture of the board (list of 9 strings)."""
    g = GoGame()
    for r, line in enumerate(rows):
        for c, ch in enumerate(line.replace(" ", "")):
            g.board[r][c] = ch
    return g

# 1. Capture: White at (0,0) loses its last liberty
g = setup(["W........", "B........"] + ["........."] * 7)
g.current_player = BLACK
res = g.play_move(0, 1)
assert res and res.captured == [(0, 0)] and g.board[0][0] == EMPTY

# 2. Suicide is illegal
g = setup([".B.......", "B........"] + ["........."] * 7)
g.current_player = WHITE
res = g.play_move(0, 0)
assert not res and res.reason == "suicide" and g.board[0][0] == EMPTY

# 3. Suicide that captures is LEGAL
g = setup([".BW......", "BW.......", "W........"] + ["........."] * 6)
g.current_player = WHITE
res = g.play_move(0, 0)   # looks like suicide, but captures (0,1) and (1,0)
assert res and set(res.captured) == {(0, 1), (1, 0)}

# 4. Ko: replay the game from your notebook. B's final recapture at (3,3) was a ko violation.
g = GoGame()
moves = [(4,4),(3,4),(2,4),(4,3),(3,3),(5,3),(3,5),(3,2),(3,6),(2,3),(5,5),(3,4)]
for m in moves:
    assert g.play_move(*m), m
res = g.play_move(3, 3)
assert not res and res.reason == "ko"
g.play_move(6, 6); g.play_move(6, 7)   # play elsewhere ("ko threat"), then retaking is fine
assert g.play_move(3, 3)

# 5. Atari
g = setup(["W........", "B........"] + ["........."] * 7)
assert g.is_in_atari(0, 0) and len(g.groups_in_atari(WHITE)) == 1

# 6. Eyes: a corner group with two real eyes
g = setup([".B.B.....",
           "BBBB.....",
           "........."] + ["........."] * 6)
group = g.get_group(1, 0)
assert g.is_eye(0, 0, BLACK) and g.is_eye(0, 2, BLACK) and g.has_two_eyes(group)
# Neither eye can be filled by White (both moves are suicide)
g.current_player = WHITE
assert not g.is_legal(0, 0) and not g.is_legal(0, 2)

# 7. False eye: enemy stone on the diagonal of an edge point
g = setup(["B.B......", "WB......."] + ["........."] * 7)
assert not g.is_eye(0, 1, BLACK)

# 8. Passing ends the game; empty board -> White wins on komi
g = GoGame()
g.pass_turn(); g.pass_turn()
assert g.game_over() and g.score() == {BLACK: 0, WHITE: 7.5} and g.winner() == WHITE

# 9. Illegal moves don't change anything
g = GoGame(); g.play_move(4, 4)
before = g.board_key()
assert not g.play_move(4, 4) and not g.play_move(9, 9) and g.board_key() == before

# 10. copy() is independent
g = GoGame(); h = g.copy(); h.play_move(4, 4)
assert g.board[4][4] == EMPTY and h.board[4][4] == BLACK

print("All tests passed")


All tests passed


In [3]:
# ---------- Stability + speed check: random games ----------
import random, time

def random_playout(game, max_moves=200):
    """Both players play random legal moves, never filling their own eyes,
    and pass when nothing else is left. (This is the core of Monte Carlo.)"""
    while not game.game_over() and len(game.move_history) < max_moves:
        player = game.current_player
        candidates = [(r, c) for r in range(game.size) for c in range(game.size)
                      if game.board[r][c] == EMPTY and not game.is_eye(r, c, player)]
        random.shuffle(candidates)
        for r, c in candidates:
            if game.play_move(r, c):
                break
        else:
            game.pass_turn()
    return game

random.seed(0)
start = time.time()
n_games = 200
black_wins = 0
for _ in range(n_games):
    g = random_playout(GoGame())
    # sanity: no group on the board may have zero liberties
    assert all(libs for _, _, libs in g.all_groups())
    black_wins += g.winner() == BLACK
elapsed = time.time() - start
print(f"{n_games} random games, no rule violations. "
      f"{n_games/elapsed:.0f} games/sec. Black won {black_wins/n_games:.0%}.")


200 random games, no rule violations. 189 games/sec. Black won 40%.


In [4]:
# ---------- Play a game by hand (type 'quit' to stop) ----------
game = GoGame()

while not game.game_over():
    game.print_board()
    print(f"Current player: {game.current_player}   captures: {game.captures}")
    move_input = input("Enter your move (row col), 'pass', or 'quit': ").strip().lower()

    if move_input == "quit":
        break
    if move_input == "pass":
        game.pass_turn()
        continue

    try:
        row, col = map(int, move_input.split())
    except ValueError:
        print("Invalid input. Please enter 'row col', 'pass', or 'quit'.")
        continue

    result = game.play_move(row, col)
    if not result:
        print(f"Illegal move ({result.reason}), try again.")
    elif result.captured:
        print(f"Captured {len(result.captured)} stone(s)!")

print("Game Over!")
game.print_board()
score = game.score()
print(f"Score (area + komi {game.komi}):  Black {score[BLACK]}  White {score[WHITE]}  ->  {game.winner()} wins")


  0 1 2 3 4 5 6 7 8
0 . . . . . . . . .
1 . . . . . . . . .
2 . . . . . . . . .
3 . . . . . . . . .
4 . . . . . . . . .
5 . . . . . . . . .
6 . . . . . . . . .
7 . . . . . . . . .
8 . . . . . . . . .

Current player: B   captures: {'B': 0, 'W': 0}
Game Over!
  0 1 2 3 4 5 6 7 8
0 . . . . . . . . .
1 . . . . . . . . .
2 . . . . . . . . .
3 . . . . . . . . .
4 . . . . . . . . .
5 . . . . . . . . .
6 . . . . . . . . .
7 . . . . . . . . .
8 . . . . . . . . .

Score (area + komi 7.5):  Black 0  White 7.5  ->  W wins


## Reward function: "Aggressive"

`compute_reward(board, move, color, strategy)` in `reward.py` scores one candidate move and returns the total plus a per-rule breakdown. Weights live in `strategies.py`: change numbers, flip `"enabled"`, or copy a block to make a new strategy. Decisions made:
- **Captures rank above liberty + connection.** One captured stone (100) beats the liberty rule (60). A capture that also touches my own stone earns both.
- **Corners get only the corner penalty** (`"include_corners": False` on the edge rule).
- **Illegal moves** score `-inf` with a reason (`"suicide"`, `"ko"`, ...).
- **Win/loss hook:** the `terminal` rule (+1 win / −1 loss, times its weight) fires on the pass that ends the game. It is off by default.

Tests: `.venv/bin/pytest` in the project folder.

In [ ]:
# ---------- Reward breakdown for three example Black moves ----------
from reward import compute_reward, format_breakdown

game = GoGame()
for r, line in enumerate([".........",
                          ".........",
                          ".........",
                          "...B.....",
                          "..BWB....",
                          "......W..",
                          "...B.B...",
                          ".........",
                          "........."]):
    game.board[r] = list(line)
game.print_board()

examples = {
    "capture the W stone at (4,3)": (5, 3),
    "press W (5,6), touching B (6,5)": (5, 5),
    "corner": (0, 0),
}
for label, move in examples.items():
    print(f"-- {label}")
    print(format_breakdown(move, compute_reward(game, move, BLACK, "aggressive")))

In [5]:
# Phase 2: Solver Aspect

  # For a 9x9 board every move will impact the whole board. First move will usually always be the center (4 4)
  # For a 11x11 board will allow for slightly more space between edge and corner moves before the fights
  # For a 19x19 board will go corners --> edge --> middle with fights where there is common interest. More long term thinking.

  # To calculate a moves impact on winpct I will need to calculate a function for influence in combination with liberties


In [6]:
# Phase 3: Monte Carlo simulation